# nb04_train : 학습 실험 (E0 스모크 → E1 기준선)

- **가속기 : GPU (P100 또는 T4 x1)** · **인터넷 : ON** (첫 실행 시 사전학습 가중치 다운로드)
- **입력** : Dataset `hms-processed`, `hms-src`
- **출력** : `/kaggle/working/models/<EXP>/` (fold별 가중치, OOF 예측, 지표) → 실행 후 Dataset `hms-models`

| 실험 | 입력 | 모델 | 가설 | 실행 설정 |
|---|---|---|---|---|
| **E0** | 10분 스펙트로그램 | EfficientNet-B0 | 파이프라인 점검 | `EXP="E0"` : fold 0, 1 epoch, 512개 |
| **E1** | 10분 스펙트로그램 | EfficientNet-B0 | 기준선 | `EXP="E1"` : 5 fold, 5 epoch, 전체 |

**실행 순서** : `EXP="E0"`로 대화형 실행(약 5분) → 오류 없으면 `EXP="E1"`로 바꿔 Save Version

## 0. 설정 (목적 : 실험 조건 고정 / 출력 : CFG, 경로)

In [1]:
EXP = "E1"            # "E0" : 스모크 테스트 / "E1" : 기준선 전체 학습

import sys, os, json, time, math
from pathlib import Path

def find_root(marker, root="/kaggle/input"):
    for pattern in [f"*/{marker}", f"*/*/{marker}", f"*/*/*/{marker}", f"*/*/*/*/{marker}"]:
        hits = sorted(Path(root).glob(pattern))
        if hits:
            return hits[0].parent
    raise FileNotFoundError(f"{marker} 를 찾지 못했습니다. Add Input 연결을 확인하세요.")

SRC_ROOT = Path(os.environ.get("HMS_SRC_DIR") or find_root("src/data/targets.py").parents[1])
PROC_DIR = Path(os.environ.get("HMS_PROC_DIR") or find_root("eeg50.npy"))
sys.path.insert(0, str(SRC_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from src.config import load_config
from src.data.targets import PROB_COLS, CLASSES
from src.data.folds import check_no_leak
from src.eval.metrics import evaluate

CFG = load_config(SRC_ROOT / "configs" / "base.yaml")
SETTINGS = {
    "E0": dict(folds=[0], epochs=1, n_limit=512),
    "E1": dict(folds=[0, 1, 2, 3, 4], epochs=5, n_limit=None),
}[EXP]
HP = dict(model="efficientnet_b0", pretrained=True, batch_size=32, lr=1e-3, weight_decay=1e-2,
          num_workers=2, seed=CFG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = Path(os.environ.get("HMS_OUT_DIR", "/kaggle/working")) / "models" / EXP
OUT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(HP["seed"]); np.random.seed(HP["seed"])
print(f"EXP={EXP} | DEVICE={DEVICE} ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'})")
print("SRC_ROOT :", SRC_ROOT, "\nPROC_DIR :", PROC_DIR, "\nOUT_DIR  :", OUT_DIR)
if DEVICE == "cpu":
    print("[경고] GPU가 꺼져 있습니다. 오른쪽 패널 Accelerator를 GPU로 바꾸세요.")

EXP=E1 | DEVICE=cuda (Tesla T4)
SRC_ROOT : /kaggle/input/datasets/sungik4057/hms-src 
PROC_DIR : /kaggle/input/datasets/sungik4057/hms-processed/hms_processed 
OUT_DIR  : /kaggle/working/models/E1


## 1. 데이터 (목적 : hms-processed 로드·검증 / 출력 : meta, spec 배열)

In [2]:
meta = pd.read_csv(PROC_DIR / "meta.csv")
spec_all = np.load(PROC_DIR / "spec10.npy", mmap_mode="r")
assert len(meta) == spec_all.shape[0], "meta.csv와 배열 행 수가 다릅니다"
assert np.allclose(meta[PROB_COLS].sum(axis=1), 1)
check_no_leak(meta)

if SETTINGS["n_limit"]:
    meta = meta.sample(SETTINGS["n_limit"], random_state=HP["seed"])
meta = meta.reset_index().rename(columns={"index": "row"})   # row : 배열에서의 위치

# 배열을 메모리에 한 번에 올림 : Dataset 디스크에서 무작위로 읽으면 매우 느림 (E0 1 epoch 155초의 원인)
t0 = time.time()
if SETTINGS["n_limit"]:
    rows = np.sort(meta["row"].to_numpy())
    SPEC = np.asarray(spec_all[rows])                          # 필요한 행만 순서대로 읽기
    meta["row"] = np.searchsorted(rows, meta["row"].to_numpy())  # 메모리 배열 기준 위치로 변경
else:
    SPEC = np.load(PROC_DIR / "spec10.npy")                    # 전체 순차 읽기 (약 4GB)
print(f"학습 대상 {len(meta):,}개 | 메모리 적재 {SPEC.shape} {SPEC.dtype} "
      f"{SPEC.nbytes / 1e9:.1f}GB, {time.time() - t0:.0f}s")
display(meta.groupby("fold").agg(n=("eeg_id", "size"), high_vote=("high_vote", "mean")).round(3))

학습 대상 17,089개 | 메모리 적재 (17089, 4, 100, 300) float16 4.1GB, 135s


,n,high_vote
fold,,
0,3345,0.361
1,3389,0.360
2,3461,0.335
3,3277,0.382
4,3617,0.304


In [3]:
class SpecDataset(torch.utils.data.Dataset):
    """10분 스펙트로그램 4부위(LL, RL, LP, RP)를 세로로 이어 붙인 1채널 이미지 (400 x 300)."""
    def __init__(self, df, arr):
        self.rows = df["row"].to_numpy()
        self.y = np.array(df[PROB_COLS].to_numpy(dtype=np.float32), copy=True)   # 쓰기 가능한 복사본 (pandas 읽기 전용 경고 방지)
        self.arr = arr                            # 메모리 배열 (워커는 fork로 복사 없이 공유)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        x = np.asarray(self.arr[self.rows[i]], dtype=np.float32).reshape(-1, 300)   # (400, 300)
        x = (x - x.mean()) / (x.std() + 1e-6)     # 표본별 표준화 : 기록 간 전력 수준 차이 보정
        return torch.from_numpy(x[None]), torch.from_numpy(self.y[i])

ds_check = SpecDataset(meta.head(4), SPEC)
xb, yb = ds_check[0]
print("입력", tuple(xb.shape), "| 타깃 합", float(yb.sum()), "| 평균·표준편차", round(float(xb.mean()), 3), round(float(xb.std()), 3))

입력 (1, 400, 300) | 타깃 합 1.0 | 평균·표준편차 -0.0 1.0


## 2. 모델·학습 함수 (목적 : 동일 조건 학습 루프 / 출력 : train_fold 함수)

In [4]:
def make_model(pretrained):
    try:
        return timm.create_model(HP["model"], pretrained=pretrained, in_chans=1, num_classes=len(CLASSES))
    except Exception as e:
        print(f"[경고] 사전학습 가중치 로드 실패 → 무작위 초기화로 진행 ({type(e).__name__}). 인터넷 설정을 확인하세요.")
        return timm.create_model(HP["model"], pretrained=False, in_chans=1, num_classes=len(CLASSES))

def kl_loss(logits, target):
    """대회 지표와 같은 방향의 손실 : KL(target || softmax(logits))."""
    return F.kl_div(F.log_softmax(logits, dim=1), target, reduction="batchmean")

@torch.no_grad()
def predict(model, loader):
    model.eval(); out = []
    for x, _ in loader:
        with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
            out.append(F.softmax(model(x.to(DEVICE)).float(), dim=1).cpu().numpy())
    return np.concatenate(out)

def train_fold(k):
    tr, va = meta[meta.fold != k], meta[meta.fold == k]
    mk = lambda d, shuffle: torch.utils.data.DataLoader(
        SpecDataset(d, SPEC), batch_size=HP["batch_size"], shuffle=shuffle,
        num_workers=HP["num_workers"], pin_memory=DEVICE == "cuda", drop_last=shuffle)
    dl_tr, dl_va = mk(tr, True), mk(va, False)

    model = make_model(HP["pretrained"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    total = SETTINGS["epochs"] * len(dl_tr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=HP["lr"], total_steps=max(total, 1), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")

    best, best_pred = float("inf"), None
    for ep in range(SETTINGS["epochs"]):
        model.train(); t0, losses = time.time(), []
        for x, y in dl_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
                loss = kl_loss(model(x).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            losses.append(loss.item())
        pred = predict(model, dl_va)
        m = evaluate(va, pred)
        print(f"  fold {k} ep {ep + 1}/{SETTINGS['epochs']} | train {np.mean(losses):.4f} | "
              f"val KL {m['kl_all']:.4f} (고득표 {m['kl_high']:.4f}) | {time.time() - t0:.0f}s", flush=True)
        if m["kl_all"] < best:                     # 검증 KL 최소 epoch 저장
            best, best_pred = m["kl_all"], pred
            torch.save(model.state_dict(), OUT_DIR / f"fold{k}.pt")
    return va.index, best_pred

## 3. 학습 실행 (목적 : fold별 학습·OOF 수집 / 출력 : fold 가중치, oof.csv)

In [5]:
oof = np.full((len(meta), len(CLASSES)), np.nan, dtype=np.float32)
t_start = time.time()
for k in SETTINGS["folds"]:
    t0 = time.time()
    idx, pred = train_fold(k)
    oof[idx] = pred
    print(f"fold {k} 완료 {(time.time() - t0) / 60:.1f}분 | 누적 {(time.time() - t_start) / 60:.1f}분", flush=True)
gpu_hours = (time.time() - t_start) / 3600
print(f"전체 학습 {gpu_hours * 60:.1f}분 (GPU {gpu_hours:.2f}시간)")

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_23/381754830.py:41: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


  fold 0 ep 1/5 | train 1.0090 | val KL 0.7460 (고득표 0.5789) | 100s
  fold 0 ep 2/5 | train 0.5446 | val KL 0.7197 (고득표 0.4277) | 88s
  fold 0 ep 3/5 | train 0.3882 | val KL 0.7103 (고득표 0.4776) | 88s
  fold 0 ep 4/5 | train 0.2159 | val KL 0.7551 (고득표 0.5568) | 88s
  fold 0 ep 5/5 | train 0.1286 | val KL 0.7551 (고득표 0.5641) | 88s
fold 0 완료 7.6분 | 누적 7.6분


/tmp/ipykernel_23/381754830.py:41: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


...

  fold 2 ep 1/5 | train 1.0939 | val KL 0.6974 (고득표 0.4605) | 92s
  fold 2 ep 2/5 | train 0.5479 | val KL 0.6618 (고득표 0.4910) | 88s
  fold 2 ep 3/5 | train 0.4075 | val KL 0.6386 (고득표 0.4554) | 88s
  fold 2 ep 4/5 | train 0.2415 | val KL 0.6618 (고득표 0.4710) | 88s
  fold 2 ep 5/5 | train 0.1448 | val KL 0.6660 (고득표 0.4760) | 88s
fold 2 완료 7.4분 | 누적 22.4분


/tmp/ipykernel_23/381754830.py:41: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


  fold 3 ep 1/5 | train 1.0306 | val KL 0.7312 (고득표 0.5582) | 93s
  fold 3 ep 2/5 | train 0.5400 | val KL 0.7202 (고득표 0.5568) | 89s
  fold 3 ep 3/5 | train 0.3975 | val KL 0.7610 (고득표 0.6100) | 89s
  fold 3 ep 4/5 | train 0.2301 | val KL 0.7699 (고득표 0.5876) | 89s
  fold 3 ep 5/5 | train 0.1356 | val KL 0.8016 (고득표 0.6067) | 89s
fold 3 완료 7.5분 | 누적 29.9분


/tmp/ipykernel_23/381754830.py:41: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


  fold 4 ep 1/5 | train 1.0042 | val KL 0.7478 (고득표 0.4686) | 92s
  fold 4 ep 2/5 | train 0.5512 | val KL 0.7277 (고득표 0.4767) | 87s
  fold 4 ep 3/5 | train 0.4063 | val KL 0.7192 (고득표 0.4914) | 87s
  fold 4 ep 4/5 | train 0.2349 | val KL 0.7845 (고득표 0.5531) | 87s
  fold 4 ep 5/5 | train 0.1395 | val KL 0.7930 (고득표 0.5748) | 87s
fold 4 완료 7.4분 | 누적 37.2분
전체 학습 37.2분 (GPU 0.62시간)


## 4. 평가·기록 (목적 : 코드북 지표 산출 / 출력 : metrics.json, results_row.csv)

In [6]:
done = ~np.isnan(oof).any(axis=1)
ev = meta[done]
metrics = evaluate(ev, oof[done], target_spec=CFG["eval"]["target_specificity"])
per_fold = {int(k): round(evaluate(ev[ev.fold == k], oof[done][(ev.fold == k).to_numpy()])["kl_all"], 4)
            for k in SETTINGS["folds"]}

pd.concat([ev[["eeg_id", "patient_id", "fold", "high_vote"] + PROB_COLS].reset_index(drop=True),
           pd.DataFrame(oof[done], columns=[f"pred_{c}" for c in CLASSES])], axis=1).to_csv(OUT_DIR / "oof.csv", index=False)

record = {"exp": EXP, "hypothesis": "baseline" if EXP == "E1" else "smoke",
          "change": "Kaggle spectrogram 4x100x300 -> EfficientNet-B0 (in_chans=1)",
          "cv_kl_all": round(metrics["kl_all"], 4), "cv_kl_high": round(metrics["kl_high"], 4),
          "public_kl": None, "private_kl": None,
          "auc_lat": round(metrics["auc_lat"], 4), "auc_pd": round(metrics["auc_pd"], 4),
          "sens_at_spec90": round(metrics["sens_at_spec90"], 4), "gpu_hours": round(gpu_hours, 2),
          "kaggle_version": None, "verdict": None, "note": f"per-fold KL {per_fold}"}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump({"record": record, "hp": HP, "settings": SETTINGS, "full": metrics}, f, indent=2, default=float)
pd.DataFrame([record]).to_csv(OUT_DIR / "results_row.csv", index=False)

display(pd.DataFrame([record]).T.rename(columns={0: EXP}))
print("저장 파일 :", sorted(p.name for p in OUT_DIR.iterdir()))

,E1
exp,E1
hypothesis,baseline
change,Kaggle spectrogram 4x100x300 -> EfficientNet-B...
cv_kl_all,0.691
cv_kl_high,0.5008
public_kl,None
private_kl,None
auc_lat,0.9121
auc_pd,0.8979
sens_at_spec90,0.7705


저장 파일 : ['fold0.pt', 'fold1.pt', 'fold2.pt', 'fold3.pt', 'fold4.pt', 'metrics.json', 'oof.csv', 'results_row.csv']


## 다음 단계
- **E0 통과 기준** : 셀 3·4가 오류 없이 끝나고 `cv_kl_all`이 숫자로 출력 (값 자체는 의미 없음)
- **E1 실행** : 셀 0의 `EXP = "E1"`, Accelerator **GPU**, Internet **ON** 확인 → Save Version (예상 1~2시간)
- **E1 완료 후** : Output으로 Dataset `hms-models` 생성, 셀 4의 결과표를 공유 → 성능 목표 재보정(M2 관문)